# 004 · Building Email Messages

> **Standard Library › Networking** · Python 3.14

The `email` package builds and parses email messages — headers, plain-text and HTML bodies, attachments, and all the MIME encoding rules in between. It is the counterpart of `System.Net.Mail.MailMessage` (or, more closely, MimeKit's `MimeMessage`). Sending is a separate job for `smtplib`.

**In this notebook you will learn:**

- creating a message with `email.message.EmailMessage`: headers and `set_content`
- adding an HTML alternative (`add_alternative`) and attachments from memory (`add_attachment`)
- serialising with `as_string()` / `as_bytes()`
- parsing messages with `message_from_string` / `BytesParser` and `policy.default`
- helpers in `email.utils`: `formataddr`, `parseaddr`, `make_msgid`, `formatdate`
- what sending with `smtplib` looks like (shown, not executed)

## .NET mental model

| .NET | Python |
|---|---|
| `MailMessage` / MimeKit `MimeMessage` | `email.message.EmailMessage` |
| `new MailAddress("alice@example.com", "Alice")` | `formataddr(("Alice", "alice@example.com"))` / `headerregistry.Address` |
| `message.Body` + `IsBodyHtml` | `msg.set_content(text)` / `subtype="html"` |
| `AlternateView` / MimeKit `BodyBuilder` (TextBody + HtmlBody) | `msg.add_alternative(html, subtype="html")` |
| `new Attachment(stream, name)` | `msg.add_attachment(bytes, maintype, subtype, filename=...)` |
| MimeKit `message.WriteTo(stream)` | `msg.as_bytes()` / `msg.as_string()` |
| MimeKit `MimeMessage.Load(stream)` | `BytesParser(policy=policy.default).parse(f)` |
| `SmtpClient.Send(message)` / MailKit `SmtpClient` | `smtplib.SMTP(...).send_message(msg)` |

Python's `email` package is a full MIME library like MimeKit — not just a thin "mail DTO" like `MailMessage`. It handles header encoding, multipart structure and transfer encodings for you.

All addresses in this notebook use the reserved `example.com` domain.

## A plain-text message

```csharp
// C#
var mail = new MailMessage
{
    From = new MailAddress("alice@example.com", "Alice Example"),
    Subject = "Weekly report",
    Body = "Hi Bob,\n\nthe report is attached.\n",
};
mail.To.Add("bob@example.com");
```

An `EmailMessage` behaves like a dict of headers plus a body. Assign headers with `msg["Name"] = value`; set the body with `set_content()`.

In [ ]:
from email.message import EmailMessage

msg = EmailMessage()
msg["From"] = "Alice Example <alice@example.com>"
msg["To"] = "bob@example.com"
msg["Subject"] = "Weekly report"
msg.set_content("Hi Bob,\n\nthe weekly report is below.\n\nAlice\n")

print(msg.as_string())

`set_content` chose `text/plain; charset="utf-8"`, a transfer encoding, and added `MIME-Version`. You never write MIME headers by hand.

⚠️ **Gotcha:** `msg["Name"] = value` **appends** a header; it never replaces one. Headers are a multi-dict, because emails legitimately repeat some headers (`Received`, custom `X-...` headers). For headers that must be unique (`Subject`, `From`, `To`, `Date`, ...) the default policy raises `ValueError` on a second assignment; for others you silently get duplicates. To change a value, `del` it first or use `replace_header`:

In [ ]:
try:
    msg["Subject"] = "Weekly report (v2)"       # Subject already exists
except ValueError as e:
    print(f"{type(e).__name__}: {e}")

msg["X-Tag"] = "finance"
msg["X-Tag"] = "weekly"                          # non-unique header → a second copy
print(msg.get_all("X-Tag"))

del msg["Subject"]                               # removes all Subject headers
msg["Subject"] = "Weekly report (v2)"
msg.replace_header("Subject", "Weekly report (final)")
print(msg["Subject"])

## Addresses, dates and message IDs with `email.utils`

```csharp
// C#
var from = new MailAddress("alice@example.com", "Alice Example");
Console.WriteLine(from.ToString());   // "Alice Example" <alice@example.com>
```

| Function | Purpose |
|---|---|
| `formataddr((name, addr))` | build `Name <addr>`, quoting/encoding the name if needed |
| `parseaddr(s)` / `getaddresses([...])` | split address header values back into `(name, addr)` pairs |
| `make_msgid(domain=...)` | a unique `Message-ID` like `<...@example.com>` |
| `formatdate(timestamp, usegmt=...)` / `format_datetime(dt)` | RFC 5322 date strings |
| `parsedate_to_datetime(s)` | parse a date header to an aware `datetime` |

In [ ]:
from email.utils import formataddr, parseaddr, getaddresses, make_msgid, formatdate, parsedate_to_datetime

print(formataddr(("Alice Example", "alice@example.com")))
print(formataddr(("Müller, Bob", "bob@example.com")))          # comma → quoted, non-ASCII → encoded
print(parseaddr("Alice Example <alice@example.com>"))
print(getaddresses(["bob@example.com, Carol <carol@example.com>"]))

msgid = make_msgid(domain="example.com")
print(msgid.startswith("<"), msgid.endswith("@example.com>"))  # random part differs every time

print(formatdate(0, usegmt=True))                               # the Unix epoch as an email date
print(parsedate_to_datetime("Thu, 01 Jan 1970 00:00:00 GMT"))

**Tip:** for the current time use `formatdate(localtime=True)` — or, with the default policy, simply assign a timezone-aware `datetime` to `msg["Date"]` and it is formatted for you.

With `policy.default` (which `EmailMessage` uses), address headers are parsed into objects — handy for reading recipients:

In [ ]:
from email.headerregistry import Address

msg2 = EmailMessage()
msg2["To"] = "bob@example.com, Carol Example <carol@example.com>"
msg2["Cc"] = Address("Dave Example", "dave", "example.com")      # display name, user, domain

for a in msg2["To"].addresses:
    print(repr(a.display_name), a.username, a.domain, "->", a.addr_spec)
print(msg2["Cc"])

## Plain text + HTML: `add_alternative`

```csharp
// C# (MimeKit)
var builder = new BodyBuilder
{
    TextBody = "Hi Bob, total: 42.50 EUR",
    HtmlBody = "<p>Hi Bob, total: <b>42.50 EUR</b></p>",
};
message.Body = builder.ToMessageBody();
```

Always send a plain-text part *and* the HTML part. `set_content(text)` followed by `add_alternative(html, subtype="html")` turns the message into `multipart/alternative`; mail clients show the richest part they support.

In [ ]:
from email.utils import formatdate

order_mail = EmailMessage()
order_mail["From"] = formataddr(("Shop Example", "orders@example.com"))
order_mail["To"] = formataddr(("Bob Example", "bob@example.com"))
order_mail["Subject"] = "Your order #1042 ✓"                    # non-ASCII is fine
order_mail["Date"] = formatdate(0, usegmt=True)                  # fixed date → repeatable output
order_mail["Message-ID"] = make_msgid(domain="example.com")

order_mail.set_content("Hi Bob,\n\nyour order #1042 has shipped. Total: 42.50 EUR\n")
order_mail.add_alternative(
    "<html><body><p>Hi Bob,</p><p>your order <b>#1042</b> has shipped. Total: <b>42.50 EUR</b></p></body></html>",
    subtype="html",
)
print(order_mail.get_content_type(), order_mail.is_multipart())

## Attachments from memory: `add_attachment`

```csharp
// C#
var bytes = Encoding.UTF8.GetBytes("sku,qty\nA-1,2\n");
mail.Attachments.Add(new Attachment(new MemoryStream(bytes), "items.csv", "text/csv"));
```

`add_attachment(data, maintype, subtype, filename=...)` takes **bytes** (or `str` for text) — no file on disk needed. The message becomes `multipart/mixed`, wrapping the alternative part plus the attachment.

In [ ]:
csv_bytes = "sku,qty\nA-1,2\nB-7,1\n".encode("utf-8")
order_mail.add_attachment(csv_bytes, maintype="text", subtype="csv", filename="items.csv")

png_header = bytes.fromhex("89504e470d0a1a0a")                 # just the 8-byte PNG signature, as demo data
order_mail.add_attachment(png_header, maintype="image", subtype="png", filename="logo.png")

for part in order_mail.walk():                                   # depth-first over all MIME parts
    print(f"{part.get_content_type():24} filename={part.get_filename()}")

**Tip:** if you don't know the MIME type of some data, `mimetypes.guess_type("report.pdf")` returns `('application/pdf', None)`; split it on `/` for `maintype`/`subtype`, and fall back to `application/octet-stream`.

## Serialising: `as_string()` and `as_bytes()`

`as_bytes()` produces the exact wire format (what you'd save as a `.eml` file or hand to an SMTP server); `as_string()` gives the same as `str`. Notice the encoded-word in `Subject` (`=?utf-8?...?=`), and the base64-encoded binary attachment.

The MIME boundaries and `Message-ID` are random, so we mask them for display:

In [ ]:
import re

raw = order_mail.as_string()
masked = re.sub(r"=+\d+==", "<BOUNDARY>", raw)
masked = re.sub(r"<\d[^@>]*@example\.com>", "<MSGID@example.com>", masked)
print(masked[:1200])

In [ ]:
wire = order_mail.as_bytes()
print(type(wire).__name__, len(wire) > 1000, wire.isascii())   # MIME output is pure ASCII

⚠️ **Gotcha:** lines in `as_string()`/`as_bytes()` end with `\n` under `policy.default`. SMTP requires `\r\n` — `smtplib.send_message` converts for you, but if you write the bytes to a socket yourself, use `msg.as_bytes(policy=policy.SMTP)`.

## Parsing messages

To read an email (e.g. a `.eml` file or bytes fetched via IMAP) use `email.message_from_bytes` / `message_from_string`, or a `BytesParser`. **Always pass `policy=policy.default`** — it gives you an `EmailMessage` with the modern API (`get_body`, `iter_attachments`, decoded headers).

In [ ]:
from email import policy, message_from_string
from email.parser import BytesParser

parsed = BytesParser(policy=policy.default).parsebytes(wire)

print(type(parsed).__name__)
print(parsed["Subject"])                       # decoded back to Unicode
print([a.addr_spec for a in parsed["To"].addresses])
print(parsed["Date"].datetime)                 # Date header is parsed to a datetime

`get_body(preferencelist=...)` picks the best body part; `iter_attachments()` yields the attachments; `get_content()` decodes a part to `str` (text) or `bytes` (binary):

In [ ]:
text_body = parsed.get_body(preferencelist=("plain",))
html_body = parsed.get_body(preferencelist=("html", "plain"))
print(text_body.get_content().strip())
print(html_body.get_content_type())

for att in parsed.iter_attachments():
    content = att.get_content()
    print(att.get_filename(), att.get_content_type(), type(content).__name__, len(content))

⚠️ **Gotcha:** without a policy, the parser uses the legacy `compat32` policy and returns an old-style `email.message.Message`: headers are **not** decoded and helpers like `get_body()` don't exist.

In [ ]:
legacy = message_from_string(raw)                              # no policy!
print(type(legacy).__name__, "|", legacy["Subject"])
print(hasattr(legacy, "get_body"))

modern = message_from_string(raw, policy=policy.default)
print(type(modern).__name__, "|", modern["Subject"])

## Sending with `smtplib` (not executed)

Sending needs a real SMTP server and credentials, so it is shown only as a code block. The pattern — connect, upgrade to TLS, log in, send — mirrors MailKit's `SmtpClient`:

```csharp
// C# (MailKit)
using var smtp = new SmtpClient();
await smtp.ConnectAsync("smtp.example.com", 587, SecureSocketOptions.StartTls);
await smtp.AuthenticateAsync("user@example.com", "app-password");
await smtp.SendAsync(message);
await smtp.DisconnectAsync(true);
```

```python
import smtplib
import ssl

context = ssl.create_default_context()

with smtplib.SMTP("smtp.example.com", 587, timeout=10) as smtp:   # placeholder host
    smtp.starttls(context=context)                                  # upgrade to TLS
    smtp.login("user@example.com", "app-password-placeholder")       # never hard-code real secrets
    smtp.send_message(order_mail)                                   # From/To taken from the headers

# Implicit TLS (port 465) instead of STARTTLS:
# with smtplib.SMTP_SSL("smtp.example.com", 465, context=context) as smtp: ...
```

- `send_message(msg)` reads the sender and recipients (`To`, `Cc`, `Bcc`) from the headers and strips `Bcc` before sending.
- Load credentials from environment variables or a secret store, never from source code.
- For local testing, run a fake SMTP server (e.g. the third-party `aiosmtpd`; the old stdlib `smtpd` module was removed in Python 3.12) or use your mail provider's sandbox.

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| New message | `new MailMessage()` / `new MimeMessage()` | `EmailMessage()` |
| Set header | `mail.Subject = ...` | `msg["Subject"] = ...` (adds; `del` first to replace) |
| Address | `new MailAddress(addr, name)` | `formataddr((name, addr))` / `Address(...)` |
| Text body | `Body = ...` | `msg.set_content(text)` |
| HTML alternative | `AlternateView` / `BodyBuilder.HtmlBody` | `msg.add_alternative(html, subtype="html")` |
| Attachment | `new Attachment(stream, name)` | `msg.add_attachment(bytes, maintype, subtype, filename=)` |
| Message-ID | `MimeUtils.GenerateMessageId()` (MimeKit) | `make_msgid(domain=...)` |
| Date header | `DateTimeOffset` → string | `formatdate(...)` / `format_datetime(dt)` |
| Serialise | `message.WriteTo(stream)` | `msg.as_bytes()` / `as_string()` |
| Parse | `MimeMessage.Load(stream)` | `BytesParser(policy=policy.default).parsebytes(b)` |
| Read body | `message.TextBody` / `HtmlBody` | `msg.get_body(preferencelist=("plain",))` |
| Attachments | `message.Attachments` | `msg.iter_attachments()` |
| Send | `SmtpClient.Send` | `smtplib.SMTP(...).send_message(msg)` |

## Exercises

1. Build an `EmailMessage` from `formataddr(("Reports", "reports@example.com"))` to `"team@example.com"` with subject `"Daily numbers"`, a plain-text body and an HTML alternative containing a `<table>`. Print its content type and the content types of all parts.
2. Attach a JSON document generated in memory (`json.dumps({"orders": 12, "revenue": 340.5})`) as `summary.json` (`application/json`). Then serialise the message with `as_bytes()`, parse it back with `BytesParser(policy=policy.default)`, and print the attachment's filename and decoded JSON.
3. Write `recipients(msg)` that returns a sorted list of all `addr_spec`s found in the `To` and `Cc` headers (either may be missing). Test it on a message with two `To` and one `Cc` address.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import json
from email import policy
from email.message import EmailMessage
from email.parser import BytesParser
from email.utils import formataddr

# 1 — text + HTML alternative
report = EmailMessage()
report["From"] = formataddr(("Reports", "reports@example.com"))
report["To"] = "team@example.com"
report["Subject"] = "Daily numbers"
report.set_content("Orders: 12\nRevenue: 340.50\n")
report.add_alternative(
    "<table><tr><td>Orders</td><td>12</td></tr><tr><td>Revenue</td><td>340.50</td></tr></table>",
    subtype="html",
)
print(report.get_content_type(), [p.get_content_type() for p in report.walk()])

# 2 — in-memory JSON attachment, round-tripped
payload = json.dumps({"orders": 12, "revenue": 340.5}).encode()
report.add_attachment(payload, maintype="application", subtype="json", filename="summary.json")

parsed = BytesParser(policy=policy.default).parsebytes(report.as_bytes())
for att in parsed.iter_attachments():
    print(att.get_filename(), json.loads(att.get_content()))

# 3 — collect recipients from optional headers
def recipients(msg):
    found = []
    for header in ("To", "Cc"):
        if msg[header] is not None:
            found.extend(a.addr_spec for a in msg[header].addresses)
    return sorted(found)

m = EmailMessage()
m["To"] = "bob@example.com, Carol <carol@example.com>"
m["Cc"] = "alice@example.com"
print(recipients(m))
```

</details>